# CellPhoneDB Circos Plot - Final version (proper circular layout)

## ✨ Features:
- ✅ **Proper circular Circos plot layout**
- ✅ Each cell type placed as a sector on the circle
- ✅ Self-loops excluded
- ✅ Each pathway represented as an individual arrow
- ✅ Collagen-related interactions only
- ✅ COL3A1-ADGRG1 highlighted with a red outline

In [ ]:
# ============================================================
# Environment setup: works on both Google Colab and local Jupyter
# ============================================================
import os
import sys

# Detect whether we are running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # NOTE: change this to wherever you keep the analysis directory on your Drive
    BASE_DIR = '/content/drive/MyDrive/scRNA_analysis'
else:
    # NOTE: set the SCRNA_BASE_DIR environment variable, or change the fallback below
    BASE_DIR = os.environ.get('SCRNA_BASE_DIR', './data')

# Derived paths used throughout the notebook
RESULTS_DIR = os.path.join(BASE_DIR, 'Results_0-1m')
FIG_DIR     = os.path.join(BASE_DIR, 'figures_Ver2')
DB_DIR      = os.path.join(BASE_DIR, 'db')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIG_DIR,     exist_ok=True)

os.chdir(BASE_DIR)
print(f'Working directory: {os.getcwd()}')


In [ ]:
import pandas as pd
import numpy as np
from pycirclize import Circos
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from collections import defaultdict

print('✓ Packages ready')

In [ ]:
# Parameter settings
INTERACTION_SCORES_PATH = os.path.join(FIG_DIR, 'statistical_analysis_interaction_scores_timecourse_7d.txt')
OUTPUT_DIR = FIG_DIR
TIMEPOINT = "7d"
SCORE_THRESHOLD = 50

collagen_genes = ["COL3A1",]


TARGET_GENE_PAIR = ("COL3A1", "ADGRG1")

LINK_COLOR = "#4A90E2"
LINK_ALPHA = 0.5
# Link width (varied dynamically based on score)
MIN_DYNAMIC_LINK_WIDTH = 0.5 # minimum dynamic link width
MAX_DYNAMIC_LINK_WIDTH = 4.0 # maximum dynamic link width

HIGHLIGHT_EDGE_COLOR = "red"
# Width of highlighted links (red outline is fixed, inner blue is dynamic)
HIGHLIGHT_EDGE_WIDTH = 6.0 # fixed width of the red outline
HIGHLIGHT_CORE_WIDTH_FACTOR = 0.9 # multiplier applied to the inner blue link width (dynamic_width * this)

print('✓ Parameter settings complete')

In [ ]:
# Cell-type color map
cluster_colors_base = {
     # ── Astro-NT (vivid colors) ──────────────────
    "Astro-NT_1": "#7B68EE",
    "Astro-NT_2": "#FF69B4",
    "Astro-NT_3": "#2FEF2F",
    "Astro-NT_4": "#DC143C",
    "Astro-NT_5": "#00CED1",
    "Astro-NT_6": "#FFD700",

    # ── ABC / VLMC (fixed) ──────────────────────────
    "ABC_1":  "#0099FF",
    "ABC_2":  "#0033CC",
    "VLMC_1": "#FF8C00",

    # ── Microglia (pastel) ───────────────────────
    "Microglia_1": "#B7E0CA",
    "Microglia_2": "#9FC982",
    "Microglia_3": "#A28FBC",
    "Microglia_4": "#C98582",
    "Microglia_5": "#C9B7E0",
    "Microglia_6": "#D8D8A5",
    "Microglia_7": "#C9B482",
    "Microglia_8": "#B2D8D8",

    # ── Lymphoid (pastel) ────────────────────────
    "Lymphoid_1": "#82C9C9",
    "Lymphoid_2": "#D8A5B0",
    "Lymphoid_3": "#C9D8A5",
    "Lymphoid_4": "#D8BFA5",

    # ── Oligo (pastel) ───────────────────────────
    "Oligo_1":  "#BFCDD8",
    "Oligo_2":  "#C5BF9F",
    "Oligo_3":  "#82C99F",
    "Oligo_4":  "#C393D1",
    "Oligo_5":  "#A5CDD8",
    "Oligo_6":  "#AAD8A5",
    "Oligo_7":  "#86A5C4",
    "Oligo_8":  "#D8A5CC",
    "Oligo_9":  "#D19D93",
    "Oligo_10": "#B9C982",
    "Oligo_11": "#A5ABD8",
    "Oligo_12": "#82C982",
    "Oligo_13": "#D8CCA5",

    # ── OPC (pastel) ─────────────────────────────
    "OPC_1": "#D8C3BF",
    "OPC_2": "#8494AD",
    "OPC_3": "#AD9D84",
    "OPC_4": "#C08AA6",

    # ── Others (pastel) ──────────────────────────
    "Endo_1":      "#ACB793",
    "Ependymal_1": "#CDDBBC",
    "Ependymal_2": "#BCE0D8",
    "Peri_1":      "#84AD90",
    "Peri_2":      "#ADBC84",
    "RO-RPA_1":    "#9FC5BD",

    # ── SPVI-SPVC (pastel) ───────────────────────
    "SPVI-SPVC_1": "#AD8492",
    "SPVI-SPVC_2": "#93B7A9",
    "SPVI-SPVC_3": "#C1A3A7",
    "SPVI-SPVC_4": "#7AB7B4",
    "SPVI-SPVC_5": "#B47AB7",
    "SPVI-SPVC_6": "#A7B47A",
    "SPVI-SPVC_7": "#7A9AB4",

    # ── SPVC (pastel) ────────────────────────────
    "SPVC_1": "#BBCBB2",
    "SPVC_2": "#99B27F",
    "SPVC_3": "#B4A3C1",
    "SPVC_4": "#E0B7D9",
    "SPVC_5": "#B77A7A",
    "SPVC_6": "#B2BCCB",

    # ── PGRN-PARN-MDRN (pastel) ──────────────────
    "PGRN-PARN-MDRN_1": "#9BA0C9",
    "PGRN-PARN-MDRN_2": "#CBB2BC",
}

prefix = f"timecourse_{TIMEPOINT}_"
sector_colors = {f"{prefix}{key}": value for key, value in cluster_colors_base.items()}

def contains_collagen_gene(gene_pair_str, collagen_genes):
    return any(gene in collagen_genes for gene in gene_pair_str.split('_'))

print('✓ Ready')

In [ ]:
# Load data & build edge list
print('Loading data...')
means = pd.read_csv(INTERACTION_SCORES_PATH, delimiter='\t')
print(f'✓ Data loaded: {means.shape}')

edges = []
valid_columns = [col for col in means.columns if col.startswith(prefix) and '|' in col]
self_loop_count = 0

for i, row in means.iterrows():
    interacting_pair = row['interacting_pair']

    if not contains_collagen_gene(interacting_pair, collagen_genes):
        continue

    for col in valid_columns:
        score = row[col]
        if not pd.isna(score) and score > SCORE_THRESHOLD:
            source, target = col.split('|')

            if source == target:
                self_loop_count += 1
                continue

            gene_set = set(interacting_pair.split('_'))
            is_highlight = gene_set == set(TARGET_GENE_PAIR)

            edges.append({
                'source': source,
                'target': target,
                'score': score,
                'gene_pair': interacting_pair,
                'highlight': is_highlight
            })

edges_df = pd.DataFrame(edges)

# --- New: filter to keep only edges whose target contains Astro-NT ---
initial_total_pathways = len(edges_df)
edges_df = edges_df[edges_df['target'].str.contains('Astro-NT')].copy()
filtered_out_count = initial_total_pathways - len(edges_df)
print(f'✓ Filtered to only Astro-NT targets: {len(edges_df)} pathways (filtered out {filtered_out_count})')
# --- end of new section ---

print(f'✓ Self-loops removed: {self_loop_count}')
print(f'✓ Total pathways: {len(edges_df)}')
print(f'✓ Highlighted: {edges_df["highlight"].sum()}')

In [ ]:
# Compute sector sizes
celltypes = sorted(set(edges_df['source']) | set(edges_df['target']))

sector_sizes = {}
for ct in celltypes:
    out_count = len(edges_df[edges_df['source'] == ct])
    in_count = len(edges_df[edges_df['target'] == ct])
    # made strictly proportional to the number of connections; the +1 is kept to avoid pycirclize index errors
    sector_sizes[ct] = (out_count + in_count)

print(f'✓ Cell types: {len(celltypes)}')

# Dictionary to keep track of positions
sector_positions = defaultdict(lambda: {'out': 0, 'in': 0})

In [ ]:
# Build the Circos plot
print('Creating Circos object...')

# Circos object (circular layout)
circos = Circos(sectors=sector_sizes, space=5)

# Draw sectors
for sector in circos.sectors:
    color = sector_colors.get(sector.name, "#CCCCCC")
    # Set r_lim so sectors are drawn as a ring on the circumference (100-110 places them outward)
    sector.rect(r_lim=(100, 110), fc=color, ec="black", lw=0.5)
    # Place text outside the ring (r=120 moves it further outward)
    sector.text(sector.name, r=120, size=7)

print('✓ Circos object created')

In [ ]:
# Draw links
print('Drawing links...')

# Define a function to scale link width based on score
min_score_for_scaling = edges_df['score'].min() if not edges_df.empty else SCORE_THRESHOLD
max_score_for_scaling = edges_df['score'].max() if not edges_df.empty else SCORE_THRESHOLD + 1

def get_scaled_lw(score):
    if max_score_for_scaling <= min_score_for_scaling: # handle zero score range
        return (MIN_DYNAMIC_LINK_WIDTH + MAX_DYNAMIC_LINK_WIDTH) / 2
    return MIN_DYNAMIC_LINK_WIDTH + \
           (score - min_score_for_scaling) / (max_score_for_scaling - min_score_for_scaling) * \
           (MAX_DYNAMIC_LINK_WIDTH - MIN_DYNAMIC_LINK_WIDTH)

# Standard links
normal_edges = edges_df[~edges_df['highlight']]
print(f'  Standard links: {len(normal_edges)}')

for idx, edge in normal_edges.iterrows():
    source = edge['source']
    target = edge['target']
    sender_color = sector_colors.get(source, LINK_COLOR) # get the sender's color

    out_pos = sector_positions[source]['out']
    sector_positions[source]['out'] += 1

    in_pos = sector_positions[target]['in']
    sector_positions[target]['in'] += 1

    # set link width based on score
    lw = get_scaled_lw(edge['score'])

    circos.link(
        (source, out_pos, out_pos + 1),
        (target, in_pos, in_pos + 1),
        color=sender_color, # use the sender's color
        alpha=LINK_ALPHA,
        lw=lw,
        direction=1,
    )

# Highlighted links (drawn twice)
highlight_edges = edges_df[edges_df['highlight']]
print(f'  Highlighted links: {len(highlight_edges)} (drawn twice)')

# Outer layer (red)
highlight_positions = []
for idx, edge in highlight_edges.iterrows():
    source = edge['source']
    target = edge['target']
    sender_color = sector_colors.get(source, LINK_COLOR) # get the sender's color

    out_pos = sector_positions[source]['out']
    sector_positions[source]['out'] += 1

    in_pos = sector_positions[target]['in']
    sector_positions[target]['in'] += 1

    highlight_positions.append({
        'source': source,
        'target': target,
        'out_pos': out_pos,
        'in_pos': in_pos,
        'score': edge['score'], # store the score
        'sender_color': sender_color # store the sender's color
    })

    # use a fixed width for the red outline
    circos.link(
        (source, out_pos, out_pos + 1),
        (target, in_pos, in_pos + 1),
        color=HIGHLIGHT_EDGE_COLOR,
        alpha=0.8,
        lw=HIGHLIGHT_EDGE_WIDTH,
        direction=1,
    )

# Inner layer (blue)
for pos in highlight_positions:
    # set the inner-link width and color based on score, the factor, and the sender's color
    lw = get_scaled_lw(pos['score']) * HIGHLIGHT_CORE_WIDTH_FACTOR
    circos.link(
        (pos['source'], pos['out_pos'], pos['out_pos'] + 1),
        (pos['target'], pos['in_pos'], pos['in_pos'] + 1),
        color=pos['sender_color'], # use the sender's color
        alpha=LINK_ALPHA,
        lw=lw,
        direction=1,
    )

print('✓ Links drawn')

In [ ]:
# Display & save
fig = circos.plotfig(figsize=(16, 16))

plt.title(
    f"Collagen-mediated Interactions ({TIMEPOINT})\n" +
    f"Each arrow = 1 pathway (Score > {SCORE_THRESHOLD}, Self-loops excluded)",
    fontsize=16, pad=20
)

legend_elements = [
    mpatches.Patch(
        facecolor=LINK_COLOR, alpha=LINK_ALPHA,
        label=f'Collagen pathways ({len(normal_edges)} arrows)'
    ),
    mpatches.Patch(
        facecolor=LINK_COLOR, edgecolor=HIGHLIGHT_EDGE_COLOR,
        linewidth=2.5, alpha=LINK_ALPHA,
        label=f'COL3A1-ADGRG1 ({len(highlight_edges)} arrows)'
    ),
]
plt.legend(handles=legend_elements, loc='upper right',
           fontsize=12, frameon=True, fancybox=True, shadow=True)

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_svg = os.path.join(OUTPUT_DIR, f"circos_{TIMEPOINT}_final.svg")
output_png = os.path.join(OUTPUT_DIR, f"circos_{TIMEPOINT}_final.png")

fig.savefig(output_svg, format="svg", dpi=300, bbox_inches='tight')
fig.savefig(output_png, format="png", dpi=300, bbox_inches='tight')

print(f'\n✓ Saved:')
print(f'  SVG: {output_svg}')
print(f'  PNG: {output_png}')

plt.show()

In [ ]:
# Summary
print("="*70)
print("Summary")
print("="*70)
print(f"Self-loops removed: {self_loop_count}")
print(f"Cell types: {len(celltypes)}")
print(f"Total pathways: {len(edges_df)}")
print(f"  - Normal: {len(normal_edges)}")
print(f"  - Highlighted: {len(highlight_edges)}")
print("="*70)
print("\n✓ Proper circular Circos plot")
print("✓ Each cell type placed as a sector on the circumference")
print("✓ Each pathway represented as an individual arrow")
print("="*70)